# ELEP DAS picks, QC'd -> PyOcto association (2-9 Hz, 3-month run)

2026-11-17. Same PyOcto association as `picking_rainier_das_pyocto_events_octo_ELEP_11102026_savethis.ipynb`, with a
pick QC added before the association. Picks are the ELEP run `.../picking-denoising/3-month-run_2-9hz_10302026`
(hourly `MORADAS_YYYYMMDD_HH.csv`, 2023-08-23 to 2023-11-21). The GaMMA counterpart, with the same QC, picks and
stations, is `gamma_ELEP_das_rainier_QC_11172026.ipynb`.

Pick QC (function `qc_elep_file`, tested in `temp_test_elep_pick_cleaning.ipynb`), per 1-minute continuous record and on
all 1107 ELEP-picked channels, before the cut to the station channels:

1. **Boundary**: drop a pick less than `edge_s = 1` s from the start or the end of its record.
2. **Coherence**: the record needs a run of at least `min_run = 7` consecutive ELEP-picked channels with a P pick, or
   with an S pick, in which neighbouring picks are at most `max_dt = 0.5` s apart. Otherwise all picks of the record
   are dropped; if it has one, all its remaining picks are kept.

Outputs (in `.../arose-scratch-rainier/pyocto_association_11172026/rainier_mora_das_elep_qc/`, with a
`_test<hours>h_<day>` suffix when `test_window` is set):
`picks_octo_elep.csv`, `pick_qc_summary.csv` (QC counts per hourly file), `events_octo_elep.csv`,
`assignments_octo_elep.csv`, `all_pick_assignments_octo_elep.csv`.

Settings:

| item | value |
| --- | --- |
| picks | ELEP `trace_p_arrival` / `trace_s_arrival`, one P and one S at most per channel per 1-minute record, after the QC above |
| stations | 320 channels: every 10th located channel of `data/Mora_channellocation.csv`, moved to the nearest ELEP channel (ELEP picked channels 2, 6, 10, ...) |
| velocity model | 1-D PyOcto table `data/vmodel/velocity_model` (IASP91), `tolerance=2.0` |
| area | lat 45-48, lon -123 to -120, depth 0-40 km, `time_before=300` |
| thresholds | `n_picks=100`, `n_p_picks=0`, `n_s_picks=0`, `n_p_and_s_picks=0` (no P/S requirement), same as the run without QC |
| other | pyocto 0.2.0 defaults |

Environment: `test_environment` conda env. The DAS clock runs ~1.9 s early (15-17 s early from about 6 Nov 2023),
so origin times are early by the same amount relative to PNSN.


In [1]:
import os
import numpy as np
import pandas as pd
from obspy import UTCDateTime

## Most recent ELEP run: hourly MORADAS_YYYYMMDD_HH.csv, 2-9 Hz, 2023-08-23 to 2023-11-21
filepath = '/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/picking-denoising/3-month-run_2-9hz_10302026/'

## Quick test: (start day, hours), e.g. ("2023-08-27", 24) for the day of the M2.9 PNSN event. None runs all 3 months.
test_window = None

out_path = ('/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/pyocto_association_11172026/rainier_mora_das_elep_qc'
            + (f'_test{test_window[1]}h_{test_window[0]}' if test_window else '') + '/')
os.makedirs(out_path, exist_ok=True)

In [2]:
### Pick QC, applied to every hourly ELEP file before the picks are used
# Each row of an ELEP file is one channel in one 1-minute continuous record (trace_start_time).
# The QC works per 1-minute record, on all 1107 ELEP-picked channels (2, 6, 10, ...), i.e. before
# the cut to the station channels:
#   1. boundary: a pick closer than edge_s to the start or the end of its record is dropped.
#   2. coherence: with the boundary picks removed, the record needs a run of at least min_run
#      consecutive ELEP-picked channels with a P pick, or with an S pick. Two neighbouring channels
#      only count as the same run when their picks are at most max_dt seconds apart (slope
#      condition: an earthquake arrival is flat across neighbouring channels, a vehicle is not).
#      A record without such a run loses all its picks; a record with one keeps all its
#      remaining P and S picks.
edge_s = 1.0      # [s]
min_run = 7       # consecutive ELEP-picked channels (7 channels ~ 270 m of cable)
max_dt = 0.5      # [s] largest pick-time difference between neighbouring channels in a run
record_s = 60.0   # [s] length of one continuous record


def qc_elep_file(path, edge_s=1.0, min_run=7, record_s=60.0, max_dt=0.5):
    """Read one hourly ELEP file and return the picks that pass the QC, with the counts.

    Returns (picks, counts). picks has one row per kept pick, on all ELEP channels:
    station (channel number), phase ('P' / 'S'), time (UTC timestamp).
    """
    df = pd.read_csv(path, usecols=['station_code', 'trace_start_time', 'trace_p_arrival', 'trace_s_arrival'], dtype=str)
    df['station'] = df['station_code'].str[len('das'):].astype(int)
    df['record'] = pd.to_datetime(df['trace_start_time'], utc=True)
    # position of the channel in the list of ELEP-picked channels (2, 6, 10, ... -> 0, 1, 2, ...)
    df['ich'] = np.searchsorted(np.sort(df['station'].unique()), df['station'])

    picks = df.melt(id_vars=['record', 'station', 'ich'], value_vars=['trace_p_arrival', 'trace_s_arrival'],
                    var_name='phase', value_name='time').dropna(subset=['time'])
    picks['phase'] = picks['phase'].map({'trace_p_arrival': 'P', 'trace_s_arrival': 'S'})
    picks['time'] = pd.to_datetime(picks['time'], utc=True)
    offset = (picks['time'] - picks['record']).dt.total_seconds()

    # 1. boundary check
    edge = (offset < edge_s) | (offset > record_s - edge_s)

    # 2. coherence check: longest flat run of consecutive channels per record, in P or in S
    good = picks[~edge].assign(offset=offset[~edge]).sort_values(['record', 'phase', 'ich'])
    new_run = ((good['ich'].diff() != 1) | (good['record'] != good['record'].shift())
               | (good['phase'] != good['phase'].shift()) | (good['offset'].diff().abs() > max_dt))
    run_len = good.groupby(new_run.cumsum())['ich'].transform('size')
    ok_records = good.loc[run_len >= min_run, 'record'].unique()
    keep = ~edge & picks['record'].isin(ok_records)

    counts = {'records': df['record'].nunique(), 'records_kept': len(ok_records), 'picks': len(picks),
              'picks_boundary': int(edge.sum()), 'picks_incoherent_record': int((~edge & ~keep).sum()),
              'picks_kept': int(keep.sum())}
    return picks.loc[keep, ['station', 'phase', 'time']], counts

In [3]:
### Stations: same layout as the PhaseNet-DAS run (every 10th located channel, 320 stations).
# ELEP only picked every 4th channel (2, 6, 10, ...), so each of those channels c is replaced by the
# nearest ELEP channel: c itself when c % 4 == 2, otherwise c + 2 (at most 2 channels, 19 m, away).
geom = pd.read_csv('data/Mora_channellocation.csv')
located = geom.dropna(subset=['Longitude [°]', 'Latitude [°]']).astype({'Channel': int}).set_index('Channel')
pn_channels = located.index[located.index % 10 == 0]
elep_channels = np.where(pn_channels % 4 == 2, pn_channels, pn_channels + 2)
elep_channels = elep_channels[np.isin(elep_channels, located.index)]

### Merge the QC'd ELEP picks from the hourly files
# One row per channel per 1-minute record, station_code "das<channel>", with at most one P time
# (trace_p_arrival) and one S time (trace_s_arrival). Each file goes through qc_elep_file on all its
# channels, and the picks that pass are then cut to the station channels.
files = sorted(f for f in os.listdir(filepath) if f.startswith('MORADAS_') and f.endswith('.csv'))
file_time = pd.to_datetime([f[len('MORADAS_'):-len('.csv')] for f in files], format='%Y%m%d_%H', utc=True)
if test_window:
    t_start = pd.Timestamp(test_window[0], tz='UTC')
    keep = (file_time >= t_start) & (file_time < t_start + pd.Timedelta(hours=test_window[1]))
    files = [f for f, k in zip(files, keep) if k]
print('%d hourly ELEP files' % len(files))

station_channels = set(elep_channels)
all_csv, qc_rows = [], []
for f in files:
    df, counts = qc_elep_file(filepath + f, edge_s, min_run, record_s, max_dt)   # picks that pass, all ELEP channels
    df = df[df['station'].isin(station_channels)]
    counts['picks_kept_on_stations'] = len(df)
    qc_rows.append(counts)
    if len(df):
        all_csv.append(df)
picks = pd.concat(all_csv, ignore_index=True)

qc_summary = pd.DataFrame(qc_rows, index=pd.Index(files, name='file'))
qc_summary.to_csv(out_path + 'pick_qc_summary.csv')
n = qc_summary.sum()
print('pick QC (edge_s=%g s, min_run=%d, max_dt=%g s), all ELEP channels:' % (edge_s, min_run, max_dt))
print('  %d of %d 1-minute records kept (%.1f%%)' % (n['records_kept'], n['records'], 100 * n['records_kept'] / n['records']))
print('  %d picks: %d dropped at a record boundary, %d dropped in records without a coherent run, %d kept (%.1f%%)' % (
      n['picks'], n['picks_boundary'], n['picks_incoherent_record'], n['picks_kept'], 100 * n['picks_kept'] / n['picks']))
print('  %d kept picks on the %d station channels' % (n['picks_kept_on_stations'], len(station_channels)))

# change to Octo format
picks['time'] = (picks['time'] - pd.Timestamp(0, tz='UTC')).dt.total_seconds()
picks['station'] = picks['station'].astype(str)   # PyOcto matches picks to the station table on this id
picks = picks.sort_values('time').reset_index(drop=True)

picks.to_csv(out_path + 'picks_octo_elep.csv', index=False)

2184 hourly ELEP files
pick QC (edge_s=1 s, min_run=7, max_dt=0.5 s), all ELEP channels:
  9913 of 126453 1-minute records kept (7.8%)
  23938406 picks: 995449 dropped at a record boundary, 20164896 dropped in records without a coherent run, 2778061 kept (11.6%)
  1011034 kept picks on the 320 station channels


In [4]:
picks

,station,phase,time
0,502,S,1.692818e+09
1,2230,S,1.692818e+09
2,1282,P,1.692818e+09
3,990,P,1.692818e+09
4,962,P,1.692818e+09
...,...,...,...
1011029,2802,S,1.700606e+09
1011030,1042,P,1.700606e+09
1011031,1042,S,1.700606e+09
1011032,2270,P,1.700606e+09


In [5]:
### Station table for PyOcto -- the 320 ELEP station channels.
# PyOcto silently ignores picks whose station id is not in the table.
stations_table = pd.DataFrame({
    'id': elep_channels.astype(str),
    'longitude': located.loc[elep_channels, 'Longitude [°]'].values,
    'latitude': located.loc[elep_channels, 'Latitude [°]'].values,
    'elevation': located.loc[elep_channels, 'Elevation [m]'].values,
})

picks = picks[picks['station'].isin(stations_table['id'])].reset_index(drop=True)
print('%d stations, %d picks (%s)' % (len(stations_table), len(picks),
      ', '.join('%s %d' % kv for kv in picks['phase'].value_counts().items())))

320 stations, 1011034 picks (P 569877, S 441157)


In [6]:
layers = pd.read_csv("data/vmodel/IASP91_1.csv").iloc[:78]
print(layers.dtypes)
print(layers.isna().sum())
print(layers.head())
print(layers.tail())
import os

import pyocto

velocity_model = pyocto.VelocityModel1D("data/vmodel/velocity_model", tolerance=2.0)

associator = pyocto.OctoAssociator.from_area(
    lat=(45, 48),
    lon=(-123, -120),
    zlim=(0, 40),
    time_before=300,
    velocity_model=velocity_model,
    n_picks=100,
    n_p_picks=0,
    n_s_picks=0,
    n_p_and_s_picks=0,
)

# Adds the x/y/z columns (km, in the associator's local frame) that associate() requires.
# This used to be safe to skip only because stations_table was loaded from the cached
# stations_table.csv, which was already transformed; the table is rebuilt from the two
# inventories on every run now, so it arrives with lon/lat/elev only.
associator.transform_stations(stations_table)
# stations_table.to_csv(filepath + 'stations_table.csv', index=False)
stations_table.head(), picks.head()

depth    float64
r        float64
vp       float64
vs       float64
dtype: object
depth    0
r        0
vp       0
vs       0
dtype: int64
   depth       r   vp    vs
0    0.0  6371.0  5.8  3.36
1    1.0  6370.0  5.8  3.36
2    2.0  6369.0  5.8  3.36
3    3.0  6368.0  5.8  3.36
4    4.0  6367.0  5.8  3.36
    depth       r      vp      vs
73  360.0  6011.0  8.8475  4.7830
74  370.0  6001.0  8.8840  4.8004
75  380.0  5991.0  8.9205  4.8178
76  390.0  5981.0  8.9570  4.8352
77  400.0  5971.0  8.9935  4.8526


(   id  longitude  latitude  elevation          x          y      z
 0   2 -121.73695  46.78581     1667.0 -18.093358  31.799013 -1.667
 1  10 -121.73703  46.78563     1668.0 -18.099527  31.779022 -1.668
 2  22 -121.73841  46.78538     1664.0 -18.204988  31.751549 -1.664
 3  30 -121.73922  46.78498     1662.0 -18.266975  31.707270 -1.662
 4  42 -121.73974  46.78407     1655.0 -18.306991  31.606229 -1.655,
   station phase          time
 0     502     S  1.692818e+09
 1    2230     S  1.692818e+09
 2    1282     P  1.692818e+09
 3     990     P  1.692818e+09
 4     962     P  1.692818e+09)

In [7]:
# Association
events, assignments = associator.associate(picks, stations_table)

print('%d events from %d picks' % (len(events), len(picks)))
if len(events):   # an empty result has no columns at all, so transform/convert would KeyError
    associator.transform_events(events)
    events['time'] = events['time'].apply(UTCDateTime)
all_pick_assignments = pd.merge(events, assignments, left_on="idx", right_on="event_idx", suffixes=("", "_pick"))

23 events from 1011034 picks


In [8]:
### Save association results (out_path is set in the first cell)
# UTCDateTime objects would be written via str(); isoformat gives a round-trippable timestamp
events_out = events.copy()
if len(events_out):
    events_out['time'] = events_out['time'].apply(lambda t: t.isoformat())
events_out.to_csv(out_path + 'events_octo_elep.csv', index=False)

assignments.to_csv(out_path + 'assignments_octo_elep.csv', index=False)   # raw pick -> event_idx table

all_out = all_pick_assignments.copy()
if len(all_out):
    all_out['time'] = all_out['time'].apply(lambda t: t.isoformat())
    all_out['time_pick'] = pd.to_datetime(all_out['time_pick'], unit='s', utc=True)   # pick epoch seconds -> ISO timestamp
all_out.to_csv(out_path + 'all_pick_assignments_octo_elep.csv', index=False)

print('saved %d events, %d assignments to %s' % (len(events_out), len(assignments), out_path))

saved 23 events, 4195 assignments to /1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/pyocto_association_11172026/rainier_mora_das_elep_qc/


In [9]:
events

,idx,time,x,y,z,picks,latitude,longitude,depth
0,0,2023-08-27T05:36:03.897274Z,-29.105575,28.665304,10.625,268,46.757232,-121.880964,10.625
1,1,2023-08-27T10:10:21.898996Z,-27.257602,20.823767,9.375,463,46.686772,-121.856312,9.375
2,2,2023-08-27T11:16:31.364092Z,-28.181589,28.665304,10.625,202,46.757271,-121.868870,10.625
3,3,2023-08-27T11:35:13.312880Z,-29.105575,29.972227,4.375,153,46.768988,-121.881047,4.375
4,4,2023-09-17T07:08:52.674939Z,44.813346,83.556060,39.375,120,47.250088,-120.908016,39.375
5,5,2023-09-25T05:34:24.885038Z,-19.865710,31.279149,0.625,195,46.781083,-121.760138,0.625
6,6,2023-09-30T13:40:05.599586Z,-29.105575,29.972227,0.625,118,46.768988,-121.881047,0.625
7,7,2023-10-09T02:21:08.217466Z,-88.240712,158.050658,23.125,102,47.915571,-122.680533,23.125
8,8,2023-10-20T17:40:09.442015Z,-30.029562,29.972227,0.625,189,46.768947,-121.893143,0.625
9,9,2023-10-23T02:42:09.755297Z,-26.333616,32.586072,0.625,129,46.792616,-121.844908,0.625
